# Financial News Sentiment Classification with ruRoBERTa

This notebook presents a transformer-based solution for financial sentiment classification in Russian-language financial news headlines and comments.

The task is based on the Kaggle competition **MEPhI M-25 ML Financial News Headlines**. The goal is to identify the financial organization mentioned in a text fragment and classify the sentiment toward that organization as negative, neutral, or positive.

The solution uses `ai-forever/ruRoBERTa-large`, multi-head classification, 5-fold cross-validation, threshold tuning, and entity-aware post-processing.

The final output is a `submission.csv` file compatible with the Kaggle submission format.

## Notebook Structure

1. Imports and configuration  
2. Reproducibility setup  
3. Data loading  
4. Text preprocessing  
5. Entity detection rules  
6. Label encoding  
7. Binary target helpers  
8. Dataset and tokenization  
9. Model architecture  
10. Training helpers  
11. Threshold tuning and post-processing  
12. Cross-validation training  
13. Out-of-fold threshold tuning  
14. Test prediction  
15. Submission generation

## 1. Imports and Configuration

This section imports the required Python libraries and defines the main configuration parameters used across the notebook.

The notebook expects the dataset files to be located in the repository `data/` directory:

```text
data/train.csv
data/test.csv
data/sample_submission.csv
```

In [ ]:
import os
import re
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42

MODEL_NAME = "ai-forever/ruRoBERTa-large"

MAX_LEN = 128

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4

EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
N_FOLDS = 5
DROPOUT = 0.20
PATIENCE = 3
N_CLASSES = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE == "cuda"

# Project paths.
# This works both when the notebook is launched from the repository root
# and when it is launched from the notebooks/ directory.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "roberta_outputs"
SUBMISSION_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = SUBMISSION_DIR / "submission.csv"

TARGETS = [
    "sber",
    "vtb",
    "gazprom",
    "alfabank",
    "raiffeisen",
    "rshb",
    "company",
]

BANK_TARGETS = [
    "sber",
    "vtb",
    "gazprom",
    "alfabank",
    "raiffeisen",
    "rshb",
]

BINARY_COLUMNS = []

for target in TARGETS:
    BINARY_COLUMNS.extend([
        f"{target}_n",
        f"{target}_0",
        f"{target}_p",
    ])

SUBMISSION_COLUMNS = ["id"] + BINARY_COLUMNS

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Epochs:", EPOCHS)
print("Folds:", N_FOLDS)

## 2. Reproducibility Setup

A fixed random seed is used to make the training process more reproducible across runs.

In [ ]:
def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 3. Data Loading

The notebook loads `train.csv`, `test.csv`, and `sample_submission.csv` from the local `data/` directory.

The raw dataset is not included in the GitHub repository. It should be downloaded from Kaggle and placed inside `data/`.

In [ ]:
def locate_file(filename: str) -> Path:
    """Locate a required dataset file inside the project data directory."""
    candidates = [
        DATA_DIR / filename,
        PROJECT_ROOT / filename,
        CURRENT_DIR / filename,
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Cannot find {filename}. Please place it in the data directory: {DATA_DIR}"
    )


TRAIN_PATH = locate_file("train.csv")
TEST_PATH = locate_file("test.csv")
SAMPLE_PATH = locate_file("sample_submission.csv")

train_df = pd.read_csv(TRAIN_PATH, sep=";")
test_df = pd.read_csv(TEST_PATH, sep=";")
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample submission path:", SAMPLE_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

assert {"id", "text"}.issubset(train_df.columns)
assert {"id", "text"}.issubset(test_df.columns)
assert list(sample_submission.columns) == SUBMISSION_COLUMNS

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

## 4. Text Preprocessing

Two text representations are created:

- `clean_text`: used as model input.
- `entity_text`: used for rule-based entity detection.

The model input is kept close to the original text because transformer models benefit from preserving linguistic context.

In [ ]:
EMOJI_RE = re.compile(
    "[\U00010000-\U0010ffff"
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF]+",
    flags=re.UNICODE,
)


def preprocess_for_model(text):
    """Clean text before passing it to the transformer model."""
    if not isinstance(text, str):
        return ""

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = EMOJI_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_for_entities(text):
    """Normalize text for rule-based entity matching."""
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.replace("#", " ")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


train_df["clean_text"] = train_df["text"].apply(preprocess_for_model)
test_df["clean_text"] = test_df["text"].apply(preprocess_for_model)

train_df["entity_text"] = train_df["text"].apply(normalize_for_entities)
test_df["entity_text"] = test_df["text"].apply(normalize_for_entities)

display(train_df[["text", "clean_text", "entity_text"]].head())

## 5. Entity Detection Rules

Regular expressions are used to detect mentions of target financial organizations.

These rules are not the main model. They are used later during post-processing to reduce false positive predictions for banks that are not explicitly mentioned in the text.

In [ ]:
ENTITY_PATTERNS = {
    "sber": r"(?:@?sberbank\w*|@?sber\w*|сбербанк\w*|сбер\w*)",
    "vtb": r"(?:@?vtb\w*|\bвтб\b)",
    "gazprom": r"(?:газпромбанк\w*|газпром\s*банк\w*|\bгпб\b|@?gazprombank\w*|gazprom\s*bank)",
    "alfabank": r"(?:@?alfa[_\-\.\s]?bank\w*|@?alfabank\w*|alfabank[_\-\.\s]?\w*|alfa[_\-\.\s]?\w*|альфа[_\-\.\s]?банк\w*|альфабанк\w*|\bальфа\b)",
    "raiffeisen": r"(?:@?raiffeisen\w*|райффайзен\w*|райф\w*)",
    "rshb": r"(?:россельхозбанк\w*|\bрсхб\b|@?rshb\w*)",
}


def add_entity_flags(df):
    """Create binary flags indicating whether each target bank is mentioned."""
    flags = pd.DataFrame(index=df.index)

    for target, pattern in ENTITY_PATTERNS.items():
        flags[f"{target}_entity"] = df["entity_text"].str.contains(
            pattern,
            regex=True,
        ).astype(int)

    return flags


train_entity_flags = add_entity_flags(train_df)
test_entity_flags = add_entity_flags(test_df)

print("Train entity mentions:")
display(train_entity_flags.sum().to_frame("entity_mentions_train"))

print("Test entity mentions:")
display(test_entity_flags.sum().to_frame("entity_mentions_test"))

## 6. Label Encoding

Each entity-specific target is encoded as a 4-class classification problem:

| Encoded label | Meaning |
|---|---|
| 0 | No relevant mention |
| 1 | Negative sentiment |
| 2 | Neutral sentiment |
| 3 | Positive sentiment |

In [ ]:
def encode_label(value):
    """Encode competition labels into four classes."""
    if pd.isna(value):
        return 0

    value = int(value)

    if value == -1:
        return 1
    if value == 0:
        return 2
    if value == 1:
        return 3

    return 0


for target in TARGETS:
    train_df[f"{target}_label"] = train_df[target].apply(encode_label)

labels_matrix = np.stack(
    [train_df[f"{target}_label"].values for target in TARGETS],
    axis=1,
)

print("Labels matrix shape:", labels_matrix.shape)

for i, target in enumerate(TARGETS):
    counts = pd.Series(labels_matrix[:, i]).value_counts().sort_index()
    print(target, counts.to_dict())

## 7. Binary Target Helpers

The Kaggle submission format requires binary columns for each entity-sentiment pair.

These helper functions convert multi-class entity predictions into the required binary format and compute the macro F1-score used for evaluation.

In [ ]:
def labels_to_binary(label_matrix, index):
    """Convert encoded entity labels into binary entity-sentiment columns."""
    result = pd.DataFrame(index=index)

    for i, target in enumerate(TARGETS):
        result[f"{target}_n"] = (label_matrix[:, i] == 1).astype(int)
        result[f"{target}_0"] = (label_matrix[:, i] == 2).astype(int)
        result[f"{target}_p"] = (label_matrix[:, i] == 3).astype(int)

    return result[BINARY_COLUMNS]


def kaggle_macro_f1(true_binary, pred_binary):
    """Compute macro-averaged F1-score using the competition binary format."""
    return f1_score(
        true_binary[BINARY_COLUMNS].values,
        pred_binary[BINARY_COLUMNS].values,
        average="macro",
        zero_division=0,
    )


true_binary_df = labels_to_binary(labels_matrix, train_df.index)

print("Positive counts:")
display(true_binary_df.sum().sort_values(ascending=False).to_frame("positive_count"))

## 8. Dataset and Tokenization

This section defines the PyTorch dataset used by the DataLoader. Each text is tokenized with the ruRoBERTa tokenizer using padding and truncation.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class BankDataset(Dataset):
    """PyTorch dataset for financial sentiment classification."""

    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }

        if "token_type_ids" in encoded:
            item["token_type_ids"] = encoded["token_type_ids"].squeeze(0)

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

## 9. Model Architecture

The model uses a shared ruRoBERTa encoder and multiple classification heads.

The shared encoder extracts contextual text representations from the input sequence. Each classification head is responsible for predicting the sentiment label for one target organization.

This design allows the model to learn common linguistic and financial patterns while keeping entity-specific predictions separated.

In [ ]:
class MultiHeadRuRoberta(nn.Module):
    """Shared ruRoBERTa encoder with one classification head per target entity."""

    def __init__(self, model_name, n_entities=7, n_classes=4, dropout=0.2):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.heads = nn.ModuleList([
            nn.Linear(hidden_size, n_classes)
            for _ in range(n_entities)
        ])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**kwargs)

        cls_vector = outputs.last_hidden_state[:, 0, :]
        cls_vector = self.dropout(cls_vector)

        logits_list = [
            head(cls_vector)
            for head in self.heads
        ]

        return logits_list

## 10. Training Helpers

The training loop uses class-weighted cross-entropy loss, gradient accumulation, gradient clipping, mixed precision on CUDA, and a cosine learning rate scheduler.

In [ ]:
def get_class_weights(labels_col):
    """Compute normalized class weights for an imbalanced target column."""
    counts = np.bincount(labels_col, minlength=N_CLASSES)
    weights = len(labels_col) / (N_CLASSES * np.maximum(counts, 1))
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float)


def train_epoch(model, loader, optimizer, scheduler, criteria, scaler):
    """Train the model for one epoch."""
    model.train()

    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits_list = model(input_ids, attention_mask, token_type_ids)

            loss = 0.0
            for i in range(len(TARGETS)):
                loss = loss + criteria[i](logits_list[i], labels[:, i])

            loss = loss / len(TARGETS)
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def predict_proba(model, loader):
    """Predict class probabilities for each target entity."""
    model.eval()

    all_probs = [[] for _ in TARGETS]

    for batch in tqdm(loader, desc="predict", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        logits_list = model(input_ids, attention_mask, token_type_ids)

        for i, logits in enumerate(logits_list):
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            all_probs[i].append(probs)

    all_probs = [
        np.vstack(entity_probs)
        for entity_probs in all_probs
    ]

    return all_probs


def probs_to_label_matrix(probs_list):
    """Convert probability arrays into predicted class labels."""
    return np.stack(
        [probs.argmax(axis=1) for probs in probs_list],
        axis=1,
    )


def probs_to_scores(probs_list, index):
    """Convert probabilities into entity-sentiment score columns."""
    scores = pd.DataFrame(index=index)

    for i, target in enumerate(TARGETS):
        probs = probs_list[i]

        scores[f"{target}_n"] = probs[:, 1]
        scores[f"{target}_0"] = probs[:, 2]
        scores[f"{target}_p"] = probs[:, 3]

    return scores[BINARY_COLUMNS]

## 11. Threshold Tuning and Post-processing

The competition expects binary predictions for each entity-sentiment pair. Instead of applying the same threshold to all output columns, this solution tunes a separate threshold for each column using out-of-fold predictions.

Entity-aware post-processing is used to reduce false positives. If a specific bank is not detected in the text, the corresponding sentiment predictions are removed.

In [ ]:
def tune_thresholds(scores_df, true_binary_df):
    """Tune one binary threshold per entity-sentiment column."""
    thresholds = {}
    column_f1 = {}

    for column in BINARY_COLUMNS:
        y_true = true_binary_df[column].values
        y_score = scores_df[column].values

        best_threshold = 0.50
        best_f1 = -1

        for threshold in np.arange(0.05, 0.96, 0.05):
            y_pred = (y_score >= threshold).astype(int)

            score = f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )

            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        thresholds[column] = best_threshold
        column_f1[column] = best_f1

    return thresholds, column_f1


def apply_thresholds(scores_df, thresholds):
    """Apply tuned thresholds to prediction scores."""
    pred = pd.DataFrame(index=scores_df.index)

    for column in BINARY_COLUMNS:
        pred[column] = (scores_df[column] >= thresholds[column]).astype(int)

    return pred[BINARY_COLUMNS]


def enforce_single_sentiment(pred_df, scores_df):
    """Ensure that each target entity has at most one active sentiment per row."""
    out = pred_df.copy()

    for target in TARGETS:
        cols = [f"{target}_n", f"{target}_0", f"{target}_p"]
        multi = out[cols].sum(axis=1) > 1

        for idx in out.index[multi]:
            best_col = scores_df.loc[idx, cols].idxmax()
            out.loc[idx, cols] = 0
            out.loc[idx, best_col] = 1

    return out[BINARY_COLUMNS]


def apply_postprocessing_rule(pred_df, data_part, rule_name):
    """Apply entity-aware post-processing rules."""
    out = pred_df.copy()
    entity_flags = add_entity_flags(data_part)

    if rule_name == "remove_all_banks_without_entity":
        for target in BANK_TARGETS:
            entity_col = f"{target}_entity"
            cols = [f"{target}_n", f"{target}_0", f"{target}_p"]

            mask = entity_flags[entity_col] == 0
            out.loc[mask, cols] = 0

        return out

    raise ValueError(f"Unknown rule: {rule_name}")

## 12. Cross-validation Training

The model is trained using stratified K-fold cross-validation. Out-of-fold predictions are saved for threshold tuning, and test predictions are averaged across folds.

In [ ]:
texts = train_df["clean_text"].tolist()
test_texts = test_df["clean_text"].tolist()

test_dataset = BankDataset(test_texts, labels=None)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=PIN_MEMORY,
)

oof_probs = [
    np.zeros((len(train_df), N_CLASSES), dtype=np.float32)
    for _ in TARGETS
]

test_probs = [
    np.zeros((len(test_df), N_CLASSES), dtype=np.float32)
    for _ in TARGETS
]

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

stratify_col = labels_matrix[:, 0]

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(texts, stratify_col), start=1):
    print("
" + "=" * 80)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("=" * 80)

    set_seed(SEED + fold)

    train_texts = [texts[i] for i in train_idx]
    valid_texts = [texts[i] for i in valid_idx]

    train_labels = labels_matrix[train_idx]
    valid_labels = labels_matrix[valid_idx]

    train_dataset = BankDataset(train_texts, labels=train_labels)
    valid_dataset = BankDataset(valid_texts, labels=valid_labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=PIN_MEMORY,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=PIN_MEMORY,
    )

    model = MultiHeadRuRoberta(
        MODEL_NAME,
        n_entities=len(TARGETS),
        n_classes=N_CLASSES,
        dropout=DROPOUT,
    ).to(DEVICE)

    criteria = []

    for i in range(len(TARGETS)):
        weights = get_class_weights(train_labels[:, i]).to(DEVICE)
        criteria.append(nn.CrossEntropyLoss(weight=weights))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
    total_steps = update_steps_per_epoch * EPOCHS

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_score = -1
    no_improve = 0
    best_path = OUTPUT_DIR / f"best_fold_{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            criteria,
            scaler,
        )

        valid_probs = predict_proba(model, valid_loader)
        valid_pred_labels = probs_to_label_matrix(valid_probs)

        true_bin = labels_to_binary(valid_labels, np.arange(len(valid_idx)))
        pred_bin = labels_to_binary(valid_pred_labels, np.arange(len(valid_idx)))

        val_f1 = kaggle_macro_f1(true_bin, pred_bin)

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"loss={loss:.4f} | "
            f"val_argmax_submission_f1={val_f1:.5f}"
        )

        if val_f1 > best_score:
            best_score = val_f1
            no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"New best fold score: {best_score:.5f}")
        else:
            no_improve += 1
            print("No improvement:", no_improve)

            if no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    fold_scores.append(best_score)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))

    valid_probs = predict_proba(model, valid_loader)

    for i in range(len(TARGETS)):
        oof_probs[i][valid_idx] = valid_probs[i]

    fold_test_probs = predict_proba(model, test_loader)

    for i in range(len(TARGETS)):
        test_probs[i] += fold_test_probs[i] / N_FOLDS

    del model, train_loader, valid_loader, train_dataset, valid_dataset
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("
Fold scores:", fold_scores)
print("Mean CV score:", np.mean(fold_scores))

## 13. Out-of-fold Threshold Tuning

Out-of-fold predictions are used to tune binary thresholds for each target column.

In [ ]:
oof_scores = probs_to_scores(oof_probs, train_df.index)
test_scores = probs_to_scores(test_probs, test_df.index)

thresholds, column_f1 = tune_thresholds(
    oof_scores,
    true_binary_df,
)

thresholds_df = pd.DataFrame({
    "column": list(thresholds.keys()),
    "threshold": list(thresholds.values()),
    "column_f1": list(column_f1.values()),
}).sort_values("column")

display(thresholds_df)

oof_pred_base = apply_thresholds(oof_scores, thresholds)
oof_pred_base = enforce_single_sentiment(oof_pred_base, oof_scores)

oof_pred_all_banks = apply_postprocessing_rule(
    oof_pred_base,
    train_df,
    "remove_all_banks_without_entity",
)

oof_base_f1 = kaggle_macro_f1(true_binary_df, oof_pred_base)
oof_all_banks_f1 = kaggle_macro_f1(true_binary_df, oof_pred_all_banks)

print("
OOF threshold F1 base:", round(oof_base_f1, 5))
print("OOF threshold F1 all_banks:", round(oof_all_banks_f1, 5))

## 14. Test Prediction

The tuned thresholds and entity-aware post-processing rules are applied to the averaged test predictions.

In [ ]:
test_pred_base = apply_thresholds(test_scores, thresholds)
test_pred_base = enforce_single_sentiment(test_pred_base, test_scores)

test_pred_all_banks = apply_postprocessing_rule(
    test_pred_base,
    test_df,
    "remove_all_banks_without_entity",
)

changed_all_banks = (
    test_pred_base[BINARY_COLUMNS].values
    != test_pred_all_banks[BINARY_COLUMNS].values
).sum()

print("Changed cells after all-banks post-processing rule:", changed_all_banks)

## 15. Submission Generation

The final predictions are converted into the exact format required by Kaggle. The generated `submission.csv` file contains one row per expected ID and binary columns for every entity-sentiment pair.

In [ ]:
def build_submission(pred_df):
    """Build a Kaggle-compatible submission file."""
    submission_from_test = pd.DataFrame({
        "id": test_df["id"].values,
    })

    for col in BINARY_COLUMNS:
        submission_from_test[col] = pred_df[col].values

    submission_from_test = submission_from_test[sample_submission.columns]

    final = sample_submission.copy()

    for col in BINARY_COLUMNS:
        final[col] = 0

    ft = submission_from_test.set_index("id")
    fn = final.set_index("id")

    common_ids = fn.index.intersection(ft.index)
    fn.loc[common_ids, BINARY_COLUMNS] = ft.loc[common_ids, BINARY_COLUMNS]

    final = fn.reset_index()
    final = final[sample_submission.columns]

    print("
Final submission checks")
    print("Rows predicted from test.csv:", submission_from_test.shape[0])
    print("Rows expected by Kaggle:", sample_submission.shape[0])
    print("Common IDs copied:", len(common_ids))
    print("Missing IDs filled with zeros:", sample_submission.shape[0] - len(common_ids))
    print("Shape:", final.shape)
    print("Columns correct:", list(final.columns) == list(sample_submission.columns))
    print("Only 0/1:", final.drop(columns=["id"]).isin([0, 1]).all().all())

    for target in TARGETS:
        cols = [f"{target}_n", f"{target}_0", f"{target}_p"]
        print(target, "max active sentiment columns:", final[cols].sum(axis=1).max())

    return final


final_submission = build_submission(test_pred_all_banks)

final_submission.to_csv(SUBMISSION_PATH, index=False)

print("
Saved submission file:")
print(SUBMISSION_PATH)

print("
Final submission positive counts:")
cols = [c for c in final_submission.columns if c != "id"]
display(
    final_submission[cols]
    .sum()
    .sort_values(ascending=False)
    .to_frame("positive_count")
)

print("
Done.")